In [0]:
# 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone
import uuid

print("Libraries imported successfully.")


# 2. Project configuration

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.data_quality_report"
)

SILVER_TABLES = {
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "products": f"{CATALOG}.{SCHEMA}.silver_products",
    "orders": f"{CATALOG}.{SCHEMA}.silver_orders",
    "order_items": f"{CATALOG}.{SCHEMA}.silver_order_items",
    "payments": f"{CATALOG}.{SCHEMA}.silver_payments",
    "shipments": f"{CATALOG}.{SCHEMA}.silver_shipments",
    "inventory": f"{CATALOG}.{SCHEMA}.silver_inventory"
}

RUN_ID = str(uuid.uuid4())

# All checks in this run share the same timestamp.

CHECKED_AT = datetime.now(timezone.utc).replace(tzinfo=None)

print("=" * 75)
print("DATA QUALITY VALIDATION CONFIGURATION")
print("=" * 75)
print(f"Catalog       : {CATALOG}")
print(f"Schema        : {SCHEMA}")
print(f"Report table  : {REPORT_TABLE}")
print(f"Run ID        : {RUN_ID}")
print(f"Checked at    : {CHECKED_AT}")
print("=" * 75)


# 3. Validation result storage

results = []

def add_result(
    dataset_name,
    check_name,
    column_name,
    total_records,
    failed_records,
    details
):

    total_records = int(total_records)
    failed_records = int(failed_records)

    failure_percentage = (
        round((failed_records / total_records) * 100, 2)
        if total_records > 0
        else 0.0
    )

    status = (
        "FAIL"
        if failed_records > 0
        else "PASS"
    )

    results.append((
        dataset_name,
        check_name,
        column_name,
        total_records,
        failed_records,
        failure_percentage,
        status,
        details,
        CHECKED_AT
    ))


# 4. Generic validation functions

def check_not_null(
    df,
    dataset_name,
    column_name,
    primary_key=False
):

    total = df.count()

    failed = (
        df.filter(
            F.col(column_name).isNull()
            | (
                F.trim(F.col(column_name).cast("string")) == ""
            )
        ).count()
    )

    check_name = (
        "PRIMARY_KEY_NOT_NULL"
        if primary_key
        else "NOT_NULL"
    )

    add_result(
        dataset_name,
        check_name,
        column_name,
        total,
        failed,
        f"Checks null or blank values in {column_name}"
    )


def check_duplicates(
    df,
    dataset_name,
    key_columns
):

    total = df.count()

    duplicate_groups = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
    )

    # Count excess duplicate rows, not duplicate groups.

    duplicate_rows = (
        duplicate_groups
        .agg(
            F.sum(F.col("count") - 1).alias("duplicates")
        )
        .first()["duplicates"]
    )

    failed = int(duplicate_rows or 0)

    add_result(
        dataset_name,
        "DUPLICATE_CHECK",
        ",".join(key_columns),
        total,
        failed,
        f"Checks duplicate records using {key_columns}"
    )


def check_condition(
    df,
    dataset_name,
    check_name,
    column_name,
    condition,
    details
):

    total = df.count()

    failed = (
        df.filter(
            condition
        ).count()
    )

    add_result(
        dataset_name,
        check_name,
        column_name,
        total,
        failed,
        details
    )


# 5. Load Silver tables

silver_data = {}

for dataset_name, table_name in SILVER_TABLES.items():

    print(f"Loading: {table_name}")

    if not spark.catalog.tableExists(table_name):

        raise Exception(
            f"Required Silver table not found: {table_name}"
        )

    df = spark.table(table_name)

    silver_data[dataset_name] = df

    print(
        f"{dataset_name}: {df.count()} records loaded"
    )

print("All Silver tables loaded successfully.")


# 6. Customer data quality checks

df = silver_data["customers"]
dataset = "customers"

print("Validating customers...")

check_not_null(df, dataset, "customer_id", True)
check_duplicates(df, dataset, ["customer_id"])

check_not_null(df, dataset, "first_name")
check_not_null(df, dataset, "last_name")
check_not_null(df, dataset, "email")

check_condition(
    df,
    dataset,
    "INVALID_EMAIL_FORMAT",
    "email",
    ~F.col("email").rlike(
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    ),
    "Checks email format"
)

check_condition(
    df,
    dataset,
    "INVALID_PHONE_FORMAT",
    "phone",
    F.col("phone").isNotNull()
    & ~F.col("phone").rlike(r"^[0-9]{10}$"),
    "Checks phone number contains exactly 10 digits"
)

check_not_null(df, dataset, "city")
check_not_null(df, dataset, "state")


# 7. Product data quality checks

df = silver_data["products"]
dataset = "products"

print("Validating products...")

check_not_null(df, dataset, "product_id", True)
check_duplicates(df, dataset, ["product_id"])

check_not_null(df, dataset, "product_name")
check_not_null(df, dataset, "category")

check_condition(
    df,
    dataset,
    "INVALID_UNIT_PRICE",
    "unit_price",
    F.col("unit_price").isNull()
    | (F.col("unit_price") <= 0),
    "Unit price must be greater than zero"
)

check_condition(
    df,
    dataset,
    "INVALID_UNIT_COST",
    "unit_cost",
    F.col("unit_cost").isNull()
    | (F.col("unit_cost") < 0),
    "Unit cost must not be negative"
)

check_condition(
    df,
    dataset,
    "INVALID_STOCK_STATUS",
    "stock_status",
    F.col("stock_status").isNull()
    | ~F.upper(F.trim(F.col("stock_status"))).isin(
        "IN STOCK",
        "OUT OF STOCK",
        "LOW STOCK",
        "IN_STOCK",
        "OUT_OF_STOCK",
        "LOW_STOCK"
    ),
    "Checks supported stock status values"
)


# 8. Order data quality checks

df = silver_data["orders"]
dataset = "orders"

print("Validating orders...")

check_not_null(df, dataset, "order_id", True)
check_duplicates(df, dataset, ["order_id"])

check_not_null(df, dataset, "customer_id")
check_not_null(df, dataset, "order_timestamp")
check_not_null(df, dataset, "order_status")
check_not_null(df, dataset, "currency")

check_condition(
    df,
    dataset,
    "INVALID_ORDER_TOTAL",
    "order_total",
    F.col("order_total").isNull()
    | (F.col("order_total") < 0),
    "Order total must not be negative"
)

check_condition(
    df,
    dataset,
    "INVALID_ORDER_STATUS",
    "order_status",
    F.col("order_status").isNull()
    | ~F.upper(F.trim(F.col("order_status"))).isin(
        "PENDING",
        "PROCESSING",
        "CONFIRMED",
        "SHIPPED",
        "DELIVERED",
        "CANCELLED",
        "CANCELED",
        "COMPLETED",
        "RETURNED",
        "REFUNDED"
    ),
    "Checks supported order status values"
)

# COMMAND ----------

# 9. Order item data quality checks

df = silver_data["order_items"]
dataset = "order_items"

print("Validating order items...")

check_not_null(df, dataset, "order_item_id", True)
check_duplicates(df, dataset, ["order_item_id"])

check_not_null(df, dataset, "order_id")
check_not_null(df, dataset, "product_id")

check_condition(
    df,
    dataset,
    "INVALID_QUANTITY",
    "quantity",
    F.col("quantity").isNull()
    | (F.col("quantity") <= 0),
    "Quantity must be greater than zero"
)

check_condition(
    df,
    dataset,
    "INVALID_UNIT_PRICE",
    "unit_price",
    F.col("unit_price").isNull()
    | (F.col("unit_price") < 0),
    "Unit price must not be negative"
)

check_condition(
    df,
    dataset,
    "INVALID_DISCOUNT_RATE",
    "discount_rate",
    F.col("discount_rate").isNull()
    | (F.col("discount_rate") < 0)
    | (F.col("discount_rate") > 1),
    "Discount rate must be between 0 and 1"
)

check_condition(
    df,
    dataset,
    "INVALID_LINE_TOTAL",
    "line_total",
    F.col("line_total").isNull()
    | (F.col("line_total") < 0),
    "Line total must not be negative"
)

# COMMAND ----------

# 10. Payment data quality checks

df = silver_data["payments"]
dataset = "payments"

print("Validating payments...")

check_not_null(df, dataset, "payment_id", True)
check_duplicates(df, dataset, ["payment_id"])

check_not_null(df, dataset, "order_id")
check_not_null(df, dataset, "payment_method")
check_not_null(df, dataset, "payment_status")

check_condition(
    df,
    dataset,
    "INVALID_PAYMENT_AMOUNT",
    "amount",
    F.col("amount").isNull()
    | (F.col("amount") < 0),
    "Payment amount must not be negative"
)

check_condition(
    df,
    dataset,
    "INVALID_PAYMENT_STATUS",
    "payment_status",
    F.col("payment_status").isNull()
    | ~F.upper(F.trim(F.col("payment_status"))).isin(
        "SUCCESS",
        "FAILED",
        "PENDING",
        "REFUNDED",
        "CANCELLED",
        "CANCELED",
        "PROCESSING"
    ),
    "Checks supported payment status values"
)



# 11. Shipment data quality checks

df = silver_data["shipments"]
dataset = "shipments"

print("Validating shipments...")

check_not_null(df, dataset, "shipment_id", True)
check_duplicates(df, dataset, ["shipment_id"])

check_not_null(df, dataset, "order_id")
check_not_null(df, dataset, "carrier")
check_not_null(df, dataset, "shipment_status")

# Supported shipment lifecycle statuses
VALID_SHIPMENT_STATUSES = [
    "PENDING",
    "PROCESSING",
    "SHIPPED",
    "IN TRANSIT",
    "IN_TRANSIT",
    "OUT FOR DELIVERY",
    "OUT_FOR_DELIVERY",
    "DELIVERED",
    "CANCELLED",
    "CANCELED",
    "RETURNED",
    "RETURN_TO_ORIGIN",
    "FAILED"
]

check_condition(
    df,
    dataset,
    "INVALID_SHIPMENT_STATUS",
    "shipment_status",
    F.col("shipment_status").isNull()
    | ~F.upper(F.trim(F.col("shipment_status"))).isin(
        *VALID_SHIPMENT_STATUSES
    ),
    "Checks supported shipment status values"
)

check_condition(
    df,
    dataset,
    "DELIVERED_BEFORE_SHIPPED",
    "delivered_date",
    F.col("delivered_date").isNotNull()
    & F.col("shipped_date").isNotNull()
    & (
        F.col("delivered_date") < F.col("shipped_date")
    ),
    "Delivery date must not precede shipped date"
)



# 12. Inventory data quality checks

df = silver_data["inventory"]
dataset = "inventory"

print("Validating inventory...")

check_not_null(df, dataset, "inventory_snapshot_id", True)
check_duplicates(df, dataset, ["inventory_snapshot_id"])

check_not_null(df, dataset, "product_id")
check_not_null(df, dataset, "snapshot_date")
check_not_null(df, dataset, "warehouse_id")

check_condition(
    df,
    dataset,
    "INVALID_QUANTITY_ON_HAND",
    "quantity_on_hand",
    F.col("quantity_on_hand").isNull()
    | (F.col("quantity_on_hand") < 0),
    "Inventory quantity must not be negative"
)


# 13. Create validation report DataFrame

report_schema = T.StructType([
    T.StructField("dataset_name", T.StringType(), False),
    T.StructField("check_name", T.StringType(), False),
    T.StructField("column_name", T.StringType(), True),
    T.StructField("total_records", T.LongType(), False),
    T.StructField("failed_records", T.LongType(), False),
    T.StructField("failure_percentage", T.DoubleType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("details", T.StringType(), True),
    T.StructField("checked_at", T.TimestampType(), False)
])

if not results:

    raise Exception("No validation results were generated.")

report_df = spark.createDataFrame(
    results,
    schema=report_schema
)


# 14. Validate generated report

invalid_results = (
    report_df
    .filter(
        ~F.col("status").isin("PASS", "FAIL")
        | (F.col("failed_records") < 0)
        | (F.col("total_records") < 0)
        | (F.col("failed_records") > F.col("total_records"))
        | (F.col("failure_percentage") < 0)
        | (F.col("failure_percentage") > 100)
    )
)

invalid_count = invalid_results.count()

if invalid_count > 0:

    display(invalid_results)

    raise Exception(
        f"Invalid validation results detected: {invalid_count}"
    )

print("Validation report checks passed.")


# 15. Save validation report with schema evolution

if not spark.catalog.tableExists(REPORT_TABLE):

    (
        report_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(REPORT_TABLE)
    )

else:

    (
        report_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(REPORT_TABLE)
    )

print(f"Validation report saved: {REPORT_TABLE}")


# 16. Latest run summary

run_summary_df = (
    report_df
    .groupBy("dataset_name")
    .agg(
        F.count("*").alias("total_checks"),

        F.sum(
            F.when(F.col("status") == "PASS", 1).otherwise(0)
        ).alias("passed_checks"),

        F.sum(
            F.when(F.col("status") == "FAIL", 1).otherwise(0)
        ).alias("failed_checks"),

        F.sum("failed_records").alias(
            "sum_of_reported_failed_records_non_distinct"
        )
    )
    .withColumn(
        "pass_percentage",
        F.round(
            F.col("passed_checks")
            / F.col("total_checks") * 100,
            2
        )
    )
    .orderBy("dataset_name")
)

print("=" * 75)
print("DATA QUALITY VALIDATION SUMMARY")
print("=" * 75)

display(run_summary_df)


# 17. Overall validation summary

overall_summary_df = (
    report_df
    .agg(
        F.count("*").alias("total_checks"),

        F.sum(
            F.when(F.col("status") == "PASS", 1).otherwise(0)
        ).alias("passed_checks"),

        F.sum(
            F.when(F.col("status") == "FAIL", 1).otherwise(0)
        ).alias("failed_checks"),

        F.sum("failed_records").alias(
            "sum_of_reported_failed_records_non_distinct"
        )
    )
    .withColumn(
        "pass_percentage",
        F.when(
            F.col("total_checks") > 0,
            F.round(
                F.col("passed_checks")
                / F.col("total_checks") * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
)

display(overall_summary_df)


# 18. Display failed checks

failed_checks_df = (
    report_df
    .filter(F.col("status") == "FAIL")
    .orderBy(
        F.desc("failed_records"),
        "dataset_name"
    )
)

print("FAILED DATA QUALITY CHECKS:")

display(failed_checks_df)


# 19. Final verification

saved_count = (
    spark.table(REPORT_TABLE)
    .filter(
        F.col("checked_at") == F.lit(CHECKED_AT)
    )
    .count()
)

generated_count = report_df.count()

if saved_count != generated_count:

    raise Exception(
        f"Report count mismatch. "
        f"Generated: {generated_count}, Saved: {saved_count}"
    )

print("=" * 75)
print("DATA QUALITY VALIDATION COMPLETED")
print("=" * 75)
print(f"Run ID             : {RUN_ID}")
print(f"Generated checks   : {generated_count}")
print(f"Saved report rows  : {saved_count}")
print(f"Report table       : {REPORT_TABLE}")
print("=" * 75)


In [0]:
print("EXISTING TABLE SCHEMA")
spark.table(REPORT_TABLE).printSchema()

print("NEW DATAFRAME SCHEMA")
report_df.printSchema()

In [0]:
from pyspark.sql import functions as F

REPORT_TABLE = "workspace.ecommerce_silver.data_quality_report"

RUN_ID = "f2003d3e-f879-43ab-8427-78b51d2cee30"

# Your current report does not persist RUN_ID.
# Use the latest checked_at timestamp instead.

report_df = spark.table(REPORT_TABLE)

latest_timestamp = (
    report_df
    .agg(F.max("checked_at").alias("latest"))
    .first()["latest"]
)

latest_run_df = report_df.filter(
    F.col("checked_at") == F.lit(latest_timestamp)
)

display(
    latest_run_df
    .groupBy("status")
    .agg(F.count("*").alias("total_checks"))
)

display(
    latest_run_df
    .filter(F.col("status") == "FAIL")
)

In [0]:
from pyspark.sql import functions as F

REPORT_TABLE = "workspace.ecommerce_silver.data_quality_report"

report_df = spark.table(REPORT_TABLE)

latest_run = report_df.agg(
    F.max("checked_at").alias("latest")
).first()["latest"]

failed_checks = (
    report_df
    .filter(
        (F.col("checked_at") == latest_run) &
        (F.col("status") == "FAIL")
    )
    .select(
        "dataset_name",
        "check_name",
        "column_name",
        "total_records",
        "failed_records",
        "failure_percentage",
        "details"
    )
)

display(failed_checks)

In [0]:
from pyspark.sql import functions as F

SHIPMENT_TABLE = "workspace.ecommerce_silver.silver_shipments"

shipments_df = spark.table(SHIPMENT_TABLE)

display(
    shipments_df
    .groupBy("shipment_status")
    .agg(
        F.count("*").alias("record_count")
    )
    .orderBy(F.desc("record_count"))
)